# LAB 3: Detección de Anomalías con Isolation Forest
**Autor:** Fray Kana Chullo (IX Ciclo - Ingeniería de Sistemas)
**Fecha:** 2026-06-30

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

## 2. Carga del Dataset

In [ ]:
# Cargar el dataset
df = pd.read_csv('network_traffic.csv')

print(f"Dimensiones: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
df.head()

## 3. Estadísticas Descriptivas

In [ ]:
df.describe()

In [ ]:
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")
print(f"\nDistribución de etiquetas:\n{df['label'].value_counts()}")

## 4. Visualización de Distribuciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['bytes_sent'].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribución de Bytes Enviados')
axes[0].set_xlabel('Bytes Enviados')
axes[0].set_ylabel('Frecuencia')

df['duration_sec'].hist(bins=50, ax=axes[1], edgecolor='black')
axes[1].set_title('Distribución de Duración (segundos)')
axes[1].set_xlabel('Duración (segundos)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('evidencias/SCR-3.1_eda.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Gráfica SCR-3.1 guardada")

## 5. Feature Engineering

In [ ]:
# Crear nuevas features
df['bytes_ratio'] = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_per_sec'] = (df['bytes_sent'] + df['bytes_recv']) / (df['duration_sec'] + 0.1)

print("✅ Nuevas features creadas:")
print("- bytes_ratio: Proporción de bytes enviados vs recibidos")
print("- bytes_per_sec: Velocidad de transferencia")

## 6. Normalización de Datos

In [ ]:
# Seleccionar features numéricas
feature_cols = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'bytes_ratio', 'bytes_per_sec']
X = df[feature_cols].copy()

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ Datos normalizados - Shape: {X_scaled.shape}")
joblib.dump(scaler, 'scaler.pkl')
print("✅ Scaler guardado como 'scaler.pkl'")

## 7. Entrenamiento del Modelo (Isolation Forest)

In [ ]:
# Entrenar Isolation Forest
model = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
model.fit(X_scaled)

# Predicciones
y_pred = model.predict(X_scaled)
y_pred_labels = ['anomaly' if p == -1 else 'normal' for p in y_pred]

# Guardar modelo
joblib.dump(model, 'modelo_anomalias.pkl')
print("✅ Modelo guardado como 'modelo_anomalias.pkl'")

## 8. Evaluación del Modelo

In [ ]:
# Etiquetas reales
y_true = df['label'].values

# Métricas
print("=== MÉTRICAS DE EVALUACIÓN ===")
print(classification_report(y_true, y_pred_labels))

precision = precision_score(y_true, y_pred_labels, pos_label='anomaly')
recall = recall_score(y_true, y_pred_labels, pos_label='anomaly')
f1 = f1_score(y_true, y_pred_labels, pos_label='anomaly')

print(f"Precisión: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

## 9. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_true, y_pred_labels, labels=['normal', 'anomaly'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['normal', 'anomaly'], 
            yticklabels=['normal', 'anomaly'])
plt.title('Matriz de Confusión - Isolation Forest')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('evidencias/SCR-3.2_metricas.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Matriz de confusión guardada como 'SCR-3.2_metricas.png'")

## 10. Top 10 Anomalías

In [ ]:
# Obtener scores de anomalía
scores = model.decision_function(X_scaled)

# Top 10 más anómalos
anomaly_indices = np.argsort(scores)[:10]
top_anomalies = df.iloc[anomaly_indices][['timestamp', 'src_ip', 'dst_ip', 'bytes_sent', 'bytes_recv', 'duration_sec', 'label']].copy()
top_anomalies['score'] = scores[anomaly_indices]

print("=== TOP 10 ANOMALÍAS ===")
top_anomalies

## 11. Exportación del Modelo

In [ ]:
# Guardar modelo y scaler
joblib.dump(model, 'modelo_anomalias.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Guardar lista de features
with open('features.txt', 'w') as f:
    for feat in feature_cols:
        f.write(f"{feat}\n")

print("✅ Modelo exportado correctamente:")
print("  - modelo_anomalias.pkl")
print("  - scaler.pkl")
print("  - features.txt")

print("\n✅ LAB 3 COMPLETADO EXITOSAMENTE")